In [1]:
from sqlalchemy import create_engine, text

import pandas as pd
from pandas import Series, DataFrame
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
engine = create_engine('mysql+mysqlconnector://root:@localhost:3306/coating_db_260618')


In [3]:
# Tabale内のカラムの名前を取得する関数
def get_column_names(table, engine):
    query = f"SHOW COLUMNS FROM {table}"
    with engine.connect() as conn:
        result = conn.execute(text(query))
        return [row[0] for row in result]
        
# SQLのエイリアスを作成する関数
def get_alias(tables, table_names, cursor):   
    select_clauses = []
    for i in range(len(tables)):
        columns = get_column_names(tables[i], cursor)
        alias_prefix = table_names[i]
        for col in columns:
            select_clauses.append(f"{alias_prefix}.{col} AS {alias_prefix}_{col}")
    select_clause = ", ".join(select_clauses)
    return select_clause

# テーブル名とエイリアスの指定
tables = ['storing_multiple_tests', 'material_compositions', 'experiments', 'storing_tests', 'fruit_details', 'wvp_tests']
table_names = ['s_mlt', 'c', 'e', 's', 'f','w']


select_clause = get_alias(tables, table_names, engine)

In [4]:
# SQLを実行して貯蔵試験のデータを取ってくる
wvp_test_query = f"""
    SELECT c.*, w.*
    FROM material_compositions c
    LEFT JOIN wvp_tests w
        ON w.composition_id = c.id;
"""
df_wvp_test = pd.read_sql_query(wvp_test_query, engine)

df_wvp_test = df_wvp_test[df_wvp_test["wvp"].notna()]


# SQLを実行して材料組成の情報をとってくる
composition_query = f"""
    SELECT *
    FROM
    material_compositions c
    RIGHT JOIN 
    materials m ON c.id = m.composition_id
    LEFT JOIN 
    material_details md ON m.material_id = md.id;
"""
df_composition = pd.read_sql_query(composition_query, engine)

material_list_query = f"""
    SELECT *
    FROM
    material_details;
"""
df_material_list = pd.read_sql_query(material_list_query, engine)
material_dict = dict(df_material_list[['name', 'charactaristic']].values)

In [5]:
df_composition = df_composition.copy()
df_composition = df_composition[['composition_id', 'material_id', 'concentration', 'name','charactaristic']]

In [6]:
df_wvp_test.to_csv('../../data/raw/wvp_test.csv', index=False)
df_composition.to_csv('../../data/raw/composition.csv', index=False)
df_material_list.to_csv('../../data/raw/material.csv', index=False)